In [ ]:
import pandas as pd
import re
import os
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path to dataset
input_folder = "/content/drive/MyDrive/audio and transcript"

# Output folder for cleaned transcripts
output_folder = "/content/drive/MyDrive/cleaned_transcripts"

os.makedirs(output_folder, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def remove_tags(text):

    text = str(text)

    # remove tags like <laugh>, <sync>, <noise>
    text = re.sub(r"<[^>]*>", " ", text)

    # remove incomplete tags like <sync
    text = re.sub(r"<\w+", " ", text)

    # remove square bracket tags like [syncing]
    text = re.sub(r"\[.*?\]", " ", text)

    return text

In [ ]:
def extract_parentheses(text):
    match = re.search(r"\((.*?)\)", text)
    if match:
        return match.group(1)
    return text

In [ ]:


filler_words = [
    "uh","um","mm","hmm","erm","ah","uhh","umm","mmm"
]

def remove_fillers(text):

  text=text.strip().lower()
  if text in filler_words:
    return ""
  return text

In [ ]:
def normalize_text(text):

    text = re.sub(r"\s+", " ", str(text))

    return text.strip()

In [ ]:
def clean_text(text):

    text = remove_tags(text)

    text = extract_parentheses(text)

    text = remove_fillers(text)

    text = normalize_text(text)

    return text

In [ ]:
files = os.listdir(input_folder)

transcript_files = [f for f in files if "transcript" in f.lower()]

print("Total transcripts:", len(transcript_files))

Total transcripts: 188


In [ ]:
for file in tqdm(transcript_files):

    file_path = os.path.join(input_folder, file)

    df = pd.read_csv(file_path, sep="\t")

    # Remove Ellie speech
    df = df[df["speaker"].str.lower().str.contains("participant")]

    # Clean text
    df["value"] = df["value"].apply(clean_text)

    # Remove empty rows
    df = df[df["value"].str.strip() != ""]

    # Save cleaned file
    output_file = file.replace(".csv", "_cleaned.csv")

    save_path = os.path.join(output_folder, output_file)

    df.to_csv(save_path, index=False)

100%|██████████| 188/188 [00:35<00:00,  5.36it/s]
